# 03 · Modelagem e validação (Etapa 3)

Compara **baseline**, **Ridge** e **árvore rasa** nos conjuntos **A** (atributos musicais), **B** (A + período + `artist_count`) e **C** (B + identidade do artista), em dois cenários de validação:

1. **Por música:** `RepeatedGroupKFold` agrupando por `song_group` (versões duplicadas nunca separam treino e teste).
2. **Por artista:** grupos = componentes conexas do grafo de colaborações (com fallback documentado se houver componente gigante).

Todo pré-processamento fica dentro de um `Pipeline`, ajustado só no treino de cada fold. Hiperparâmetros por **CV aninhada**. Alvo: `log1p(streams)`.

> O conjunto C mede o quanto saber *quem* é o artista ajuda; **não é efeito causal da fama**.

In [1]:
import json, textwrap, warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

try:
    from IPython.display import display
except ImportError:
    display = print
warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = ROOT / "dataset" / "processed"
FIG_DIR, TABLES_DIR = ROOT / "reports" / "figures", ROOT / "reports" / "tables"
MODELS_DIR, DOCS_DIR = ROOT / "reports" / "models", ROOT / "docs"
for p in (FIG_DIR, TABLES_DIR, MODELS_DIR, DOCS_DIR):
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
N_SPLITS, N_REPEATS, INNER_SPLITS = 5, 3, 3
MIN_ARTIST_SONGS = 3          # artistas com menos músicas no treino viram "outros" (conjunto C)
GIANT_COMPONENT_SHARE = 0.20  # acima disso a CV por artista com componentes é inviável (5 folds)
PCT = ["danceability_%", "valence_%", "energy_%", "acousticness_%", "instrumentalness_%", "liveness_%", "speechiness_%"]
FEATURES = {"A": ["bpm", "key", "mode"] + PCT}
FEATURES["B"] = FEATURES["A"] + ["release_period", "artist_count"]
FEATURES["C"] = FEATURES["B"] + ["main_artist"]   # C só depois de fechar A e B
NUMERIC = ["bpm", "artist_count"] + PCT
CATEG = ["key", "mode", "release_period"]
TARGET = "log_streams"

df = pd.read_csv(PROCESSED_DIR / "tracks_clean.csv", dtype={"key": str, "mode": str, "release_period": str})
d = df[df["usable_for_supervised"]].copy().reset_index(drop=True)
y = d[TARGET].to_numpy()
print(f"{len(d)} linhas para modelagem; {d['main_artist'].nunique()} artistas principais")
CAPTIONS = []
def save_fig(fig, name, caption):
    fig.tight_layout(rect=[0, 0.07, 1, 1])
    fig.text(0.01, 0.01, textwrap.fill(caption, 150), fontsize=9, va="bottom")
    fig.savefig(FIG_DIR / name, dpi=130); plt.show(); CAPTIONS.append(dict(figura=name, legenda=caption)); print(name, "->", caption)

ModuleNotFoundError: No module named 'joblib'

## 1. Grupos de validação
**Cenário 1:** `song_group`. **Cenário 2:** grafo de colaborações (artistas ligados quando aparecem na mesma música), agrupado por componentes conexas (union-find). Antes de usar, mede-se a maior componente: com artistas muito conectados ela pode virar uma componente gigante. Se passar do limite, o notebook cai para agrupar só por `main_artist` e registra isso.

In [ ]:
parent = {}
def find(a):
    parent.setdefault(a, a)
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra

artist_lists = d["artists_list"].map(json.loads)
for lst in artist_lists:
    for a in lst: find(a)
    for a in lst[1:]: union(lst[0], a)
d["artist_component"] = [find(lst[0]) for lst in artist_lists]

comp_sizes = d["artist_component"].value_counts()
giant_share = comp_sizes.iloc[0] / len(d)
print(f"{comp_sizes.size} componentes; a maior cobre {comp_sizes.iloc[0]} músicas ({100*giant_share:.1f}%)")
if giant_share > GIANT_COMPONENT_SHARE:
    ARTIST_GROUP_MODE = "main_artist"
    d["group_artist"] = d["main_artist"].astype(str)
    print("-> componente gigante: CV por artista usa main_artist (colaboradores podem aparecer no treino e no teste).")
else:
    ARTIST_GROUP_MODE = "componentes"
    d["group_artist"] = d["artist_component"]
    print("-> CV por artista usa componentes conexas.")
GROUPS = {"musica": d["song_group"].to_numpy(), "artista": d["group_artist"].to_numpy()}

## 2. Divisões (semente fixa, salvas em arquivo)
Grupos inteiros vão para o mesmo fold, balanceados por tamanho. As mesmas divisões valem para todos os modelos e conjuntos.

In [ ]:
def make_splits(groups, n_splits, n_repeats, seed):
    sizes = pd.Series(groups).value_counts()
    fold_of = pd.Series(groups)
    out = []
    for r in range(n_repeats):
        order = sizes.sample(frac=1, random_state=seed + r).sort_values(ascending=False, kind="stable")
        load = np.zeros(n_splits); assign = {}
        for g, sz in order.items():
            k = int(np.argmin(load)); assign[g] = k; load[k] += sz
        fold = fold_of.map(assign).to_numpy()
        for k in range(n_splits):
            out.append((r, k, np.where(fold != k)[0], np.where(fold == k)[0]))
    return out

SPLITS = {sc: make_splits(g, N_SPLITS, N_REPEATS, SEED) for sc, g in GROUPS.items()}
for sc, sp in SPLITS.items():                                   # nenhuma música/artista no treino e no teste
    for r, k, tr, te in sp:
        assert not set(GROUPS[sc][tr]) & set(GROUPS[sc][te]), f"vazamento de grupo ({sc}, rep {r}, fold {k})"
json.dump({sc: [dict(rep=r, fold=k, train_row_id=d.loc[tr, "row_id"].tolist(), test_row_id=d.loc[te, "row_id"].tolist())
                for r, k, tr, te in sp] for sc, sp in SPLITS.items()},
          open(TABLES_DIR / "cv_folds.json", "w"))
sizes = pd.DataFrame([dict(cenario=sc, rep=r, fold=k, n_treino=len(tr), n_teste=len(te)) for sc, sp in SPLITS.items() for r, k, tr, te in sp])
display(sizes.groupby("cenario")["n_teste"].describe())

## 3. Pipelines e modelos
`RareToOther` implementa a regra do conjunto C: só artistas com pelo menos *k* músicas **no treino** são codificados; os demais (e qualquer artista nunca visto) caem em `"outros"`.

In [ ]:
class RareToOther(BaseEstimator, TransformerMixin):
    def __init__(self, min_count=3):
        self.min_count = min_count
    def fit(self, X, y=None):
        vc = pd.Series(np.asarray(X).ravel().astype(str)).value_counts()
        self.keep_ = set(vc[vc >= self.min_count].index)
        return self
    def transform(self, X):
        s = pd.Series(np.asarray(X).ravel().astype(str))
        return s.where(s.isin(self.keep_), "outros").to_frame().to_numpy()

def make_pipeline(feats, model):
    num = [f for f in feats if f in NUMERIC]
    cat = [f for f in feats if f in CATEG]
    parts = []
    if num: parts.append(("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num))
    if cat: parts.append(("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                                           ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat))
    if "main_artist" in feats:
        parts.append(("art", Pipeline([("rare", RareToOther(MIN_ARTIST_SONGS)),
                                       ("ohe", OneHotEncoder(handle_unknown="ignore"))]), ["main_artist"]))
    return Pipeline([("prep", ColumnTransformer(parts)), ("model", model)])

MODELS = {
    "ridge": (Ridge(), {"model__alpha": np.logspace(-2, 3, 11)}),
    "arvore": (DecisionTreeRegressor(random_state=SEED),
               {"model__max_depth": [2, 3, 4, 5], "model__min_samples_leaf": [5, 10, 20, 40]}),
}

## 4. Avaliação com CV aninhada
Para cada fold externo, a busca de hiperparâmetros usa `GroupKFold` interno **só dentro do treino**. Salva as previsões fora da amostra para reanálise sem retreinar.

In [ ]:
def metrics(yt, yp):
    return dict(mae=mean_absolute_error(yt, yp), r2=r2_score(yt, yp),
                spearman=stats.spearmanr(yt, yp)[0] if np.std(yp) > 0 else np.nan)

rows, preds, unk = [], [], []
for sc, sp in SPLITS.items():
    for r, k, tr, te in sp:
        Xtr, Xte, ytr, yte = d.iloc[tr], d.iloc[te], y[tr], y[te]
        base = DummyRegressor(strategy="median").fit(Xtr[["bpm"]], ytr)       # mediana de log_streams do treino
        runs = [("baseline", "-", base.predict(Xte[["bpm"]]), None)]
        for exp, feats in FEATURES.items():
            for mname, (est, grid) in MODELS.items():
                gs = GridSearchCV(make_pipeline(feats, est), grid, cv=GroupKFold(INNER_SPLITS),
                                  scoring="neg_mean_absolute_error", n_jobs=1)
                gs.fit(Xtr[feats], ytr, groups=GROUPS[sc][tr])
                runs.append((mname, exp, gs.predict(Xte[feats]), gs.best_params_))
        for mname, exp, yp, bp in runs:
            rows.append(dict(cenario=sc, rep=r, fold=k, modelo=mname, conjunto=exp, n_teste=len(te),
                             best_params=json.dumps(bp, default=float) if bp else "", **metrics(yte, yp)))
            preds.append(pd.DataFrame(dict(cenario=sc, rep=r, fold=k, modelo=mname, conjunto=exp,
                                           row_id=Xte["row_id"].to_numpy(), y_true=yte, y_pred=yp)))
        # artistas do teste que o conjunto C não conhece (define quanto o cenário 2 mede)
        cnt = Xtr["main_artist"].value_counts()
        known = Xte["main_artist"].isin(cnt[cnt >= MIN_ARTIST_SONGS].index)
        unk.append(dict(cenario=sc, rep=r, fold=k, n_teste=len(te), pct_teste_artista_desconhecido=100 * (1 - known.mean()),
                        n_artistas_teste=Xte["main_artist"].nunique(),
                        n_artistas_teste_desconhecidos=int((~Xte["main_artist"].drop_duplicates().isin(cnt[cnt >= MIN_ARTIST_SONGS].index)).sum())))
res = pd.DataFrame(rows)
pd.concat(preds, ignore_index=True).to_csv(TABLES_DIR / "predictions.csv", index=False)
unk = pd.DataFrame(unk); unk.to_csv(TABLES_DIR / "tab_unknown_artists.csv", index=False)
res.to_csv(TABLES_DIR / "cv_results_by_fold.csv", index=False)
print(len(res), "avaliações; previsões salvas em predictions.csv")
display(unk.groupby("cenario")[["pct_teste_artista_desconhecido", "n_artistas_teste_desconhecidos"]].mean().round(1))

## 5. Tabela comparativa e ganho sobre o baseline

In [ ]:
base_fold = res[res.modelo == "baseline"].set_index(["cenario", "rep", "fold"])["mae"]
res["mae_baseline"] = [base_fold[(c, r, k)] for c, r, k in zip(res.cenario, res.rep, res.fold)]
res["ganho_mae"] = res["mae_baseline"] - res["mae"]
res["ganho_mae_pct"] = 100 * res["ganho_mae"] / res["mae_baseline"]

def ms(s): return f"{s.mean():.3f} ± {s.std():.3f}"
comp = (res.groupby(["modelo", "conjunto", "cenario"])
           .agg(MAE_log=("mae", ms), R2_log=("r2", ms), Spearman=("spearman", ms),
                ganho_MAE_pct=("ganho_mae_pct", ms), mae_medio=("mae", "mean")).reset_index()
           .sort_values(["cenario", "modelo", "conjunto"]))
comp.drop(columns="mae_medio").to_csv(TABLES_DIR / "model_comparison.csv", index=False)
display(comp.drop(columns="mae_medio"))

fig, ax = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
for a, sc in zip(ax, ["musica", "artista"]):
    sub = res[res.cenario == sc].assign(modelo_conj=lambda t: t.modelo + " " + t.conjunto)
    sns.boxplot(data=sub, x="modelo_conj", y="mae", ax=a, color="#8172B2"); a.set_title(f"CV por {sc}")
    a.tick_params(axis="x", rotation=45); a.set_xlabel(""); a.set_ylabel("MAE em log1p(streams) por fold")
best = comp.loc[comp[comp.modelo != "baseline"].groupby("cenario")["mae_medio"].idxmin()]
save_fig(fig, "fig_mae_folds.png", f"{N_SPLITS} folds x {N_REPEATS} repetições, n={len(d)}. Menor MAE médio: " +
         "; ".join(f"{r.cenario}: {r.modelo} {r.conjunto} ({r.mae_medio:.3f})" for r in best.itertuples()) + ".")

g = res[res.modelo != "baseline"].groupby(["cenario", "modelo", "conjunto"])["ganho_mae_pct"].mean().reset_index()
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=g, x="conjunto", y="ganho_mae_pct", hue=g.cenario + " / " + g.modelo, ax=ax)
ax.axhline(0, color="k", lw=1); ax.set_ylabel("ganho de MAE sobre o baseline (%)")
save_fig(fig, "fig_gain_baseline.png",
         "Ganho percentual de MAE sobre a mediana do treino (média dos folds). Valores perto de 0 ou negativos indicam pouco sinal preditivo, resultado válido.")

## 6. Comparação pareada entre conjuntos (A vs B, B vs C)
Diferença de MAE por (repetição, fold); positivo = o conjunto mais rico erra menos. Intervalo por bootstrap sobre os folds (aproximado: folds repetidos não são independentes).

In [ ]:
rng = np.random.default_rng(SEED)
pair = []
for sc in ["musica", "artista"]:
    for m in MODELS:
        w = res[(res.cenario == sc) & (res.modelo == m)].pivot_table(index=["rep", "fold"], columns="conjunto", values="mae")
        for a, b in [("A", "B"), ("B", "C")]:
            dif = (w[a] - w[b]).to_numpy()
            bs = [rng.choice(dif, len(dif)).mean() for _ in range(2000)]
            pair.append(dict(cenario=sc, modelo=m, comparacao=f"{a} - {b}", dif_mae_media=dif.mean(),
                             ci_low=np.percentile(bs, 2.5), ci_high=np.percentile(bs, 97.5),
                             melhora=bool(np.percentile(bs, 2.5) > 0)))
pair = pd.DataFrame(pair); pair.to_csv(TABLES_DIR / "paired_comparison.csv", index=False)
display(pair.round(4))

## 7. Árvore final (insumo da Etapa 4) e protocolo de validação
A árvore é treinada nos dados completos com o conjunto B **apenas** como insumo para a Etapa 4. Não é resultado de desempenho.

In [ ]:
gs = GridSearchCV(make_pipeline(FEATURES["B"], DecisionTreeRegressor(random_state=SEED)), MODELS["arvore"][1],
                  cv=GroupKFold(INNER_SPLITS), scoring="neg_mean_absolute_error")
gs.fit(d[FEATURES["B"]], y, groups=GROUPS["musica"])
joblib.dump(gs.best_estimator_, MODELS_DIR / "tree_full_B.joblib")
print("Árvore salva; melhores parâmetros:", gs.best_params_)

u = unk.groupby("cenario")[["pct_teste_artista_desconhecido", "n_artistas_teste_desconhecidos"]].mean()
protocol = f"""# Protocolo de validação (Etapa 3)

## Dados e alvo
- {len(d)} músicas com `streams` válido (`usable_for_supervised`); alvo `log1p(streams)`.
- Conjuntos: A = atributos musicais; B = A + `release_period` + `artist_count`; C = B + `main_artist`.
- Fora de todos os modelos: `track_name`, playlists e charts.

## Divisões
- Semente {SEED}; {N_SPLITS} folds x {N_REPEATS} repetições; mesmas divisões para todos os modelos e conjuntos. Índices em `reports/tables/cv_folds.json`.
- **Cenário 1 (música):** grupo = `song_group` (título + artistas normalizados). Versões duplicadas nunca ficam em treino e teste ao mesmo tempo.
- **Cenário 2 (artista):** a maior componente conexa do grafo de colaborações cobre {100*giant_share:.1f}% das músicas ({comp_sizes.size} componentes).
  Modo usado: **{ARTIST_GROUP_MODE}**.{" Como a componente é gigante, o agrupamento é só por `main_artist`: colaboradores podem aparecer no treino e no teste, então o cenário 2 é mais otimista do que um corte total por artista." if ARTIST_GROUP_MODE == "main_artist" else ""}

## Pré-processamento e hiperparâmetros
- Imputação, normalização e codificação dentro do `Pipeline`, ajustadas só no treino de cada fold.
- Ridge: alpha em 11 valores log-espaçados; árvore: `max_depth` 2-5 e `min_samples_leaf` 5-40. Busca por CV aninhada (`GroupKFold({INNER_SPLITS})` dentro do treino).

## Conjunto C e artistas novos
- Só artistas com pelo menos {MIN_ARTIST_SONGS} músicas no treino são codificados; os demais e qualquer artista nunca visto caem em "outros".
- Fração média das músicas de teste com artista "desconhecido": música {u.loc['musica','pct_teste_artista_desconhecido']:.1f}%, artista {u.loc['artista','pct_teste_artista_desconhecido']:.1f}%.
  Esse número define quanto o conjunto C consegue de fato medir em cada cenário.

## Métricas
- MAE e R² em log, Spearman entre previsão e alvo, por fold; média ± desvio e ganho de MAE sobre o baseline (mediana do treino).
- Comparação pareada A vs B e B vs C com intervalo (`paired_comparison.csv`). Previsões fora da amostra em `predictions.csv`.

## Limitações
- C mede associação com a identidade do artista, **não** efeito causal da fama.
- `streams` é acumulado e a amostra são músicas já populares; pouco sinal preditivo é um resultado válido.
- Intervalos pareados tratam os folds como independentes, o que é aproximado.
"""
(DOCS_DIR / "validation_protocol.md").write_text(protocol, encoding="utf-8")
pd.DataFrame(CAPTIONS).to_csv(TABLES_DIR / "figure_captions_modelagem.csv", index=False)
print(protocol)